In [ ]:
# install libraries
!pip install earthengine-api geemap --quiet

In [ ]:
import ee
import geemap

# Authenticate
ee.Authenticate()
# Initialize with your specific project
ee.Initialize(project='urban-green-cover')

In [ ]:
def mask_s2_clouds(image):

    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0)\
            .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))

    return image.updateMask(mask).divide(10000)

In [ ]:
def get_s2_quality_mosaic(aoi, year_start, year_end):

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(f'{year_start}-11-01', f'{year_end}-03-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
        .map(mask_s2_clouds)
        .map(lambda img: img.addBands(
            img.normalizedDifference(['B8','B4']).rename('NDVI')
        ))
    )

    composite = (
        collection
        .qualityMosaic('NDVI')
        .select(['B2','B3','B4','B8'])
        .clip(aoi)
    )

    return composite

In [ ]:
def get_s1_median(aoi, year):

    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(f'{year}-01-01', f'{year}-12-31')
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH'))
    )

    return (
        collection.median()
        .select(['VV','VH'])
        .clip(aoi)
    )

In [ ]:
def build_change_label(aoi, start_year=22, end_year=23, forest_threshold=30):

    gfc = ee.Image("UMD/hansen/global_forest_change_2024_v1_12").clip(aoi)

    tree_cover = gfc.select('treecover2000')
    loss_year = gfc.select('lossyear')

    # Forest baseline mask
    forest_mask = tree_cover.gte(forest_threshold)

    # Forest loss within selected year range
    loss_mask = loss_year.gte(start_year).And(loss_year.lte(end_year))

    change_label = (
        loss_mask
        .And(forest_mask)
        .rename('label')
        .uint8()
    )

    return change_label

In [ ]:
def build_siamese_labels(aoi, start_year=22, end_year=23, forest_threshold=30):
    gfc = ee.Image("UMD/hansen/global_forest_change_2024_v1_12").clip(aoi)

    # 1. Pixels that were actually forest in 2021
    # (Forest in 2000 AND no loss recorded through 2021)
    forest_at_start = gfc.select('treecover2000').gte(forest_threshold) \
                      .And(gfc.select('lossyear').neq(0).And(gfc.select('lossyear').lt(22)).Not())

    # 2. Change Label (Loss in 22 or 23)
    loss_22_23 = gfc.select('lossyear').gte(22).And(gfc.select('lossyear').lte(23))

    # 3. Final Labeling
    # We only care about pixels that were forest in 2021.
    # Everything else (water, old loss, cities) is masked out (transparent).
    label = loss_22_23.updateMask(forest_at_start).rename('label').uint8()

    return label

In [ ]:
def get_reference_projection(aoi):

    proj = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .first()
        .select('B2')
        .projection()
    )

    return proj

In [ ]:
def build_siamese_stack(aoi):

    # Sentinel-2
    before_s2 = get_s2_quality_mosaic(aoi, 2020, 2021)
    after_s2  = get_s2_quality_mosaic(aoi, 2023, 2024)

    # Sentinel-1
    before_s1 = get_s1_median(aoi, 2021)
    after_s1  = get_s1_median(aoi, 2024)

    before_stack = before_s2.addBands(before_s1)
    after_stack  = after_s2.addBands(after_s1)

    change_label = build_siamese_labels(aoi, start_year=22, end_year=23)

    # Reference grid
    proj = get_reference_projection(aoi)

    before_stack = before_stack.reproject(proj)
    after_stack  = after_stack.reproject(proj)
    change_label = change_label.reproject(proj)

    before_stack = before_stack.rename([
        'B2_t1','B3_t1','B4_t1','B8_t1','VV_t1','VH_t1'
    ])

    after_stack = after_stack.rename([
        'B2_t2','B3_t2','B4_t2','B8_t2','VV_t2','VH_t2'
    ])

    siamese_stack = (
        ee.Image.cat([
            before_stack,
            after_stack,
            change_label
        ])
        .float()
        .unmask(0)
        .clip(aoi)
    )

    return siamese_stack

In [ ]:
def export_siamese_tiles(aoi, tile_size=256,
                         folder="Forest_Project_Data",
                         description="Siamese_Tiles"):

    stack = build_siamese_stack(aoi)

    task = ee.batch.Export.image.toDrive(
        image=stack,
        description=description,
        folder=folder,
        region=aoi,
        scale=10,
        fileDimensions=[tile_size, tile_size],
        fileFormat="GeoTIFF",
        formatOptions={'cloudOptimized': True},
        maxPixels=1e13
    )

    task.start()

    print("Export started.")
    return task

In [ ]:
def get_aoi_area_km2(aoi):

    area_m2 = aoi.area()
    area_km2 = area_m2.divide(1e6)

    return area_km2.getInfo()

In [ ]:
def get_tile_area_km2(tile_size=256, resolution=10):

    side_km = (tile_size * resolution) / 1000
    area_km2 = side_km ** 2

    return area_km2

In [ ]:
def estimate_tile_count(aoi, tile_size=256, resolution=10):

    area_km2 = get_aoi_area_km2(aoi)
    tile_area = get_tile_area_km2(tile_size, resolution)

    tiles = area_km2 / tile_area

    print("AOI area (km²):", area_km2)
    print("Tile area (km²):", tile_area)
    print("Estimated tiles:", int(tiles))

    return tiles

In [ ]:
def estimate_dataset_size(num_tiles,
                          tile_size=256,
                          bands=13,
                          bytes_per_pixel=4):

    pixels = tile_size * tile_size * bands
    bytes_per_tile = pixels * bytes_per_pixel

    total_bytes = bytes_per_tile * num_tiles
    total_gb = total_bytes / (1024**3)

    print("Estimated dataset size:", round(total_gb,2), "GB")

    return total_gb

In [ ]:
# Load India districts (FAO GAUL)
districts = ee.FeatureCollection("FAO/GAUL/2015/level2")
# Filter Dima Hasao district
aoi = districts.filter(
    ee.Filter.And(
        ee.Filter.eq("ADM0_NAME", "India"),
        ee.Filter.eq("ADM1_NAME", "Meghalaya"),
        # ee.Filter.eq("ADM2_NAME", "Dhalai")
    )
).geometry()

In [ ]:
# # Coordinates: [Longitude, Latitude]
# aoi = ee.Geometry.Rectangle([93.0, 25.4, 93.2, 25.6]) # for testing

In [ ]:
# Estimate tiles
tiles = estimate_tile_count(aoi)

# Estimate dataset size
estimate_dataset_size(tiles)

AOI area (km²): 22620.81778805137
Tile area (km²): 6.5536
Estimated tiles: 3451
Estimated dataset size: 10.95 GB


10.954984696383319

In [ ]:
# Export dataset
task = export_siamese_tiles(aoi,
                            folder="Meghalaya_22_23")

Export started.


In [ ]:
task.status()

{'state': 'RUNNING',
 'description': 'Siamese_Tiles',
 'priority': 100,
 'creation_timestamp_ms': 1772666174501,
 'update_timestamp_ms': 1772666333439,
 'start_timestamp_ms': 1772666184927,
 'task_type': 'EXPORT_IMAGE',
 'attempt': 1,
 'batch_eecu_usage_seconds': 192.731,
 'id': 'PPROSAOCQJ5FUOUMS3VA3PZW',
 'name': 'projects/urban-green-cover/operations/PPROSAOCQJ5FUOUMS3VA3PZW'}